# 03 — Baseline: Naive Persistence & Ridge Regression
Run **after** `01_Feature_Engineering.ipynb`.

**Input:** `forecasting_features.parquet`
**Output:** `baseline_results.csv`, `baseline_summary.csv`

**Target framing: 20-business-day horizon total**, per (Reference, Size) — the demand weight
the GA slotting stage will actually consume. Every model in this project is scored on exactly
this target.

**Why Ridge instead of a naive/statistical baseline (e.g. SBA/Croston):** SBA is built to
forecast a flat daily rate step-by-step; summing that rate across a 20-day window doesn't play
to its strength and produced a misleading, unstable result when tested (R² deeply negative).
**Ridge Regression, trained on the identical engineered features as LightGBM/XGBoost**, is a
fairer and more standard comparison: same information, same rolling-origin training, just a
linear model instead of a tree ensemble. This directly answers "does the nonlinear model
actually add value?" — the question a reviewer will ask anyway. Naive persistence is kept
alongside it purely as a simple, zero-effort reference point.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
sns.set_style('whitegrid')

panel = pd.read_parquet('forecasting_features.parquet')
panel['unique_id'] = panel['Reference'].astype(str) + "_" + panel['Size (US)'].astype(str)
biz_days = np.sort(panel['date'].unique())
H = 20
final_cutoff_idx = len(biz_days) - H
print("Final cutoff date:", pd.Timestamp(biz_days[final_cutoff_idx]).date())

feature_cols = ['lag_1','lag_2','lag_3','lag_5','lag_10','lag_20',
                 'rmean_5','rmean_10','rmean_20','rstd_5','rstd_10','rstd_20',
                 'freq_10','freq_20','freq_60','nz_mean_20','expanding_mean',
                 'days_since_last','series_age','dow','weekofyear','month','dayofmonth',
                 'ABCCOD_code','Size (US)']  # Sector_code excluded: constant across all products (no predictive value)

In [ ]:
import os as _os
# ── Image output folder ─────────────────────────────────────────────────────
# All figures will be saved to a subfolder called "figures" inside the
# directory where you run this notebook.  Change FIGURES_DIR if you prefer
# a different path.
FIGURES_DIR = _os.path.join(_os.getcwd(), "figures")
_os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"[INFO] Figures will be saved to: {FIGURES_DIR}")


## 1. Build rolling-origin training snapshots + final held-out test snapshot
A single row per combo isn't enough to *train* a model without leaking the answer, so several
historical cutoffs during training each get paired with their own "next 20 business days"
total. The final, untouched cutoff is reserved purely for testing.

In [ ]:
def build_snapshot(cutoff_idx, panel):
    feat_date = biz_days[cutoff_idx-1]
    target_days = biz_days[cutoff_idx: cutoff_idx+H]
    feats = panel[panel.date==feat_date][['unique_id']+feature_cols]
    tgt = (panel[panel.date.isin(target_days)]
           .groupby('unique_id')['picks'].sum().rename('horizon_total'))
    out = feats.merge(tgt, on='unique_id', how='left')
    out['horizon_total'] = out['horizon_total'].fillna(0)
    return out

train_cutoffs = list(range(40, final_cutoff_idx, H))
train_data = pd.concat([build_snapshot(c, panel) for c in train_cutoffs], ignore_index=True)
test_data  = build_snapshot(final_cutoff_idx, panel)
Xtr, ytr = train_data[feature_cols], train_data['horizon_total']
Xte, yte = test_data[feature_cols], test_data['horizon_total']
print("Train rows:", Xtr.shape, "| Test rows:", Xte.shape)

## 2. Naive persistence (simple reference)
Forecast = the combo's own total picks over the *previous* 20 business days.

In [ ]:
prev_days = biz_days[final_cutoff_idx-H: final_cutoff_idx]
naive_pred = (panel[panel.date.isin(prev_days)].groupby('unique_id')['picks'].sum()
              .reindex(test_data['unique_id']).fillna(0).values)
naive_mae  = mean_absolute_error(yte, naive_pred)
naive_rmse = np.sqrt(mean_squared_error(yte, naive_pred))
naive_r2   = r2_score(yte, naive_pred)
print(f"Naive -> MAE {naive_mae:.3f}  RMSE {naive_rmse:.3f}  R2 {naive_r2:.3f}")

## 3. Ridge Regression baseline (main baseline model)

In [ ]:
scaler = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)

ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(Xtr_s, ytr)
ridge_pred = np.clip(ridge.predict(Xte_s), 0, None)

ridge_mae  = mean_absolute_error(yte, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(yte, ridge_pred))
ridge_r2   = r2_score(yte, ridge_pred)
print(f"Ridge -> MAE {ridge_mae:.3f}  RMSE {ridge_rmse:.3f}  R2 {ridge_r2:.3f}")

## 4. Baseline comparison table

In [ ]:
baseline_summary = pd.DataFrame({
    'Model': ['Naive Persistence', 'Ridge Regression'],
    'MAE':  [naive_mae, ridge_mae],
    'RMSE': [naive_rmse, ridge_rmse],
    'R2':   [naive_r2, ridge_r2],
})
baseline_summary

## 5. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))

axes[0].scatter(yte, ridge_pred, alpha=0.4, s=15, color='#4C72B0')
lim = max(yte.max(), ridge_pred.max())
axes[0].plot([0,lim],[0,lim], 'k--', linewidth=1)
axes[0].set_xlabel('Actual 20-day total picks'); axes[0].set_ylabel('Ridge predicted')
axes[0].set_title('Ridge Regression: Predicted vs Actual')

sns.barplot(x='Model', y='MAE', data=baseline_summary, hue='Model', ax=axes[1], palette=['#DD8452','#4C72B0'], legend=False)
axes[1].set_title('Baseline MAE Comparison (lower is better)')
plt.tight_layout(); plt.savefig(_os.path.join(FIGURES_DIR, "03_ridge_predicted_vs_actual.png"), dpi=150, bbox_inches="tight")
plt.show()

### Ridge coefficients (which features drive the linear baseline)

In [ ]:
coef = pd.Series(ridge.coef_, index=feature_cols).sort_values()
plt.figure(figsize=(8,7))
coef.plot(kind='barh', color=['#C44E52' if v<0 else '#4C72B0' for v in coef.values])
plt.title('Ridge Regression Coefficients (standardized features)')
plt.axvline(0, color='k', linewidth=0.8)
plt.tight_layout(); plt.savefig(_os.path.join(FIGURES_DIR, "03_ridge_coefficients.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Save results

In [ ]:
results = test_data[['unique_id']].copy()
results['actual_total'] = yte.values
results['naive_pred'] = naive_pred
results['ridge_pred'] = ridge_pred
results.to_csv('baseline_results.csv', index=False)
baseline_summary.to_csv('baseline_summary.csv', index=False)
print("Saved: baseline_results.csv, baseline_summary.csv")

In [ ]:
# ── Extra graphic: Prediction error violin by ABC class ───────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd, numpy as np, os as _os

error_df = test_data[['unique_id']].copy()
error_df['ridge_error'] = ridge_pred - yte.values
error_df = error_df.merge(
    panel[['unique_id','ABCCOD']].drop_duplicates('unique_id'), on='unique_id', how='left')

plt.figure(figsize=(8,5))
sns.violinplot(data=error_df, x='ABCCOD', y='ridge_error',
               palette='Set2', inner='quartile', order=sorted(error_df['ABCCOD'].dropna().unique().astype(str)))
plt.axhline(0, color='black', linewidth=1.2, linestyle='--')
plt.title('Ridge Prediction Errors by ABC Class', fontweight='bold')
plt.xlabel('ABC Class'); plt.ylabel('Predicted − Actual')
plt.tight_layout()
plt.savefig(_os.path.join(FIGURES_DIR, "03_ridge_error_by_abc.png"), dpi=150, bbox_inches="tight")
plt.show()
